In [1]:
"""Cell 1: Imports and Paths"""
import sqlite3
import os
import math
from pathlib import Path

REPO_ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path.cwd().parent
DB_PATH = REPO_ROOT / "databases" / "berkeley_housing_v2.db"
OUTPUT_DIR = REPO_ROOT / "docs" / "tours"

In [2]:
"""Cell 2: SQL Query"""
QUERY = """
SELECT
    p.id AS project_id,
    p.canonical_name,
    p.canonical_address,
    p.latitude,
    p.longitude,
    pv.total_units AS unit_count
FROM projects p
JOIN project_versions pv ON pv.project_id = p.id AND pv.is_current = 1
WHERE pv.total_units > 200
  AND p.latitude IS NOT NULL
  AND p.longitude IS NOT NULL
  AND p.id NOT IN (165, 170, 171, 177)  -- Exclude UC student housing
ORDER BY p.latitude DESC;
"""

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()
cursor.execute(QUERY)
projects = [dict(row) for row in cursor.fetchall()]
conn.close()

print(f"Found {len(projects)} projects with >200 units")
print("\nFirst 5 projects (northernmost first):")
for p in projects[:5]:
    print(f"  {p['canonical_address']}: {p['unit_count']} units ({p['latitude']:.4f}, {p['longitude']:.4f})")

Found 17 projects with >200 units

First 5 projects (northernmost first):
  1899 OXFORD St: 212 units (37.8745, -122.2661)
  1750 SACRAMENTO St: 739 units (37.8743, -122.2830)
  1974 SHATTUCK Ave: 599 units (37.8726, -122.2688)
  2131 University Ave: 205 units (37.8725, -122.2681)
  2029 UNIVERSITY Ave: 240 units (37.8723, -122.2698)


In [3]:
"""Cell 3: Tour Parameters"""
ORBIT_RADIUS_M = 200
ORBIT_DURATION_SEC = 10
WAYPOINTS_PER_ORBIT = 12
CAMERA_ALTITUDE_M = 100
CAMERA_TILT_DEGREES = 65
GEOMETRY_KML_HREF = "../geometry.kml"

In [4]:
"""Cell 4: compute_camera_position()"""
def compute_camera_position(center_lat, center_lon, radius_m, heading_deg):
    """
    Return (cam_lat, cam_lon) for camera positioned at heading around center.
    
    The camera is placed at radius_m from the center, at the given heading.
    Heading 0 = North, 90 = East, 180 = South, 270 = West.
    The camera will look back toward the center (heading points inward).
    """
    meters_per_deg_lat = 111320
    meters_per_deg_lon = 111320 * math.cos(math.radians(center_lat))
    
    # Camera is positioned OPPOSITE to where it's looking
    # If heading is 0 (looking north toward center), camera is south of center
    # So we offset by heading + 180 degrees
    offset_heading_rad = math.radians(heading_deg + 180)
    
    # Calculate offset in meters, then convert to degrees
    delta_north_m = radius_m * math.cos(offset_heading_rad)
    delta_east_m = radius_m * math.sin(offset_heading_rad)
    
    cam_lat = center_lat + (delta_north_m / meters_per_deg_lat)
    cam_lon = center_lon + (delta_east_m / meters_per_deg_lon)
    
    return (cam_lat, cam_lon)

In [5]:
"""Cell 5: emit_flyto_camera()"""
def emit_flyto_camera(cam_lat, cam_lon, altitude_m, heading_deg, tilt_deg, duration_sec):
    """
    Return KML string for one <gx:FlyTo> with <Camera>.
    """
    return f"""    <gx:FlyTo>
      <gx:duration>{duration_sec}</gx:duration>
      <gx:flyToMode>smooth</gx:flyToMode>
      <Camera>
        <longitude>{cam_lon}</longitude>
        <latitude>{cam_lat}</latitude>
        <altitude>{altitude_m}</altitude>
        <heading>{heading_deg}</heading>
        <tilt>{tilt_deg}</tilt>
        <roll>0</roll>
        <altitudeMode>relativeToGround</altitudeMode>
      </Camera>
    </gx:FlyTo>
"""

In [6]:
"""Cell 6: generate_tour()"""
def generate_tour(query_results, orbit_radius_m, orbit_duration_sec, waypoints_per_orbit,
                  camera_altitude_m, camera_tilt_degrees, geometry_kml_href):
    """
    Generate full KML tour string with orbits around each project.
    Includes NetworkLink to geometry KML for building footprints.
    """
    step_duration = orbit_duration_sec / waypoints_per_orbit
    
    # KML header with namespaces
    kml = """<?xml version="1.0" encoding="UTF-8"?>
<kml xmlns="http://www.opengis.net/kml/2.2"
     xmlns:gx="http://www.google.com/kml/ext/2.2">
<Document>
  <name>Berkeley Large Housing Projects Tour</name>
  <description>Orbit tour of {count} private housing projects with over 200 units</description>
  
  <NetworkLink>
    <name>Project Geometries</name>
    <Link>
      <href>{geometry_href}</href>
    </Link>
  </NetworkLink>
  
  <gx:Tour>
    <name>Large Projects Orbit Tour</name>
    <gx:Playlist>
""".format(count=len(query_results), geometry_href=geometry_kml_href)
    
    # Generate orbit for each project
    for project in query_results:
        lat = project["latitude"]
        lon = project["longitude"]
        
        for i in range(waypoints_per_orbit):
            heading = (i * 360 / waypoints_per_orbit) % 360
            cam_lat, cam_lon = compute_camera_position(lat, lon, orbit_radius_m, heading)
            kml += emit_flyto_camera(cam_lat, cam_lon, camera_altitude_m, heading,
                                     camera_tilt_degrees, step_duration)
    
    # KML footer
    kml += """    </gx:Playlist>
  </gx:Tour>
</Document>
</kml>
"""
    
    return kml

In [7]:
"""Cell 7: Execute and Save"""
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

kml_content = generate_tour(
    projects,
    orbit_radius_m=ORBIT_RADIUS_M,
    orbit_duration_sec=ORBIT_DURATION_SEC,
    waypoints_per_orbit=WAYPOINTS_PER_ORBIT,
    camera_altitude_m=CAMERA_ALTITUDE_M,
    camera_tilt_degrees=CAMERA_TILT_DEGREES,
    geometry_kml_href=GEOMETRY_KML_HREF
)

output_path = OUTPUT_DIR / "tour-private-pipeline-over-200-units-2026-05-16.kml"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(kml_content)

# Summary
flyto_count = len(projects) * WAYPOINTS_PER_ORBIT
file_size = output_path.stat().st_size
total_duration = len(projects) * ORBIT_DURATION_SEC

print(f"Tour saved to: {output_path}")
print(f"Projects: {len(projects)}")
print(f"FlyTo elements: {flyto_count}")
print(f"Total duration: {total_duration} seconds ({total_duration/60:.1f} minutes)")
print(f"File size: {file_size:,} bytes")

Tour saved to: /Users/johngage/berkeley-data/docs/tours/tour-private-pipeline-over-200-units-2026-05-16.kml
Projects: 17
FlyTo elements: 204
Total duration: 170 seconds (2.8 minutes)
File size: 86,143 bytes
